In [57]:
import pandas as pd
df = pd.read_csv("C:/Users/91898/Desktop/e/spam_ham_dataset.csv")
print(df.head())
print(df.shape)

   Unnamed: 0 label                                               text  \
0         605   ham  Subject: enron methanol ; meter # : 988291\r\n...   
1        2349   ham  Subject: hpl nom for january 9 , 2001\r\n( see...   
2        3624   ham  Subject: neon retreat\r\nho ho ho , we ' re ar...   
3        4685  spam  Subject: photoshop , windows , office . cheap ...   
4        2030   ham  Subject: re : indian springs\r\nthis deal is t...   

   label_num  
0          0  
1          0  
2          0  
3          1  
4          0  
(5171, 4)


In [58]:
print(df.info())
print(df['label'].value_counts())
print(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5171 entries, 0 to 5170
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Unnamed: 0  5171 non-null   int64 
 1   label       5171 non-null   object
 2   text        5171 non-null   object
 3   label_num   5171 non-null   int64 
dtypes: int64(2), object(2)
memory usage: 161.7+ KB
None
label
ham     3672
spam    1499
Name: count, dtype: int64
Unnamed: 0    0
label         0
text          0
label_num     0
dtype: int64


In [59]:
df = df.drop(columns=['Unnamed: 0'])

In [60]:
df.head()

,label,text,label_num
0,ham,Subject: enron methanol ; meter # : 988291\r\n...,0
1,ham,"Subject: hpl nom for january 9 , 2001\r\n( see...",0
2,ham,"Subject: neon retreat\r\nho ho ho , we ' re ar...",0
3,spam,"Subject: photoshop , windows , office . cheap ...",1
4,ham,Subject: re : indian springs\r\nthis deal is t...,0


In [61]:
#cleaniing text
import re
import string
def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+',' ',text)
    text = text.translate(
        str.maketrans(' ',' ',string.punctuation)
    )
    text = re.sub(r'\d+',' ',text)
    text = " ".join(text.split())
    return text

In [62]:
df['clean_text'] = df['text'].apply(clean_text)

In [63]:
print(df[['text', 'clean_text']].head())

                                                text  \
0  Subject: enron methanol ; meter # : 988291\r\n...   
1  Subject: hpl nom for january 9 , 2001\r\n( see...   
2  Subject: neon retreat\r\nho ho ho , we ' re ar...   
3  Subject: photoshop , windows , office . cheap ...   
4  Subject: re : indian springs\r\nthis deal is t...   

                                          clean_text  
0  subject enron methanol meter this is a follow ...  
1  subject hpl nom for january see attached file ...  
2  subject neon retreat ho ho ho we re around to ...  
3  subject photoshop windows office cheap main tr...  
4  subject re indian springs this deal is to book...  


In [64]:
X = df['clean_text']
y = df['label_num']

In [65]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(
     stop_words='english',
    ngram_range=(1,2)
)
X = vectorizer.fit_transform(X)

In [66]:
print(X)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 683548 stored elements and shape (5171, 263581)>
  Coords	Values
  (0, 225756)	0.019682209540861424
  (0, 79680)	0.04453631995251731
  (0, 150052)	0.10984095605748494
  (0, 149293)	0.05706744204631223
  (0, 92578)	0.08834201717784444
  (0, 162620)	0.08086626765102023
  (0, 98631)	0.11138545083561432
  (0, 153556)	0.08191623442142455
  (0, 183404)	0.11489376987644145
  (0, 91885)	0.06928786247095886
  (0, 58950)	0.09072263180059906
  (0, 189039)	0.09425496818368748
  (0, 58396)	0.051424304369876064
  (0, 170232)	0.1496852621370099
  (0, 181310)	0.12627161926632893
  (0, 57824)	0.1452637018279128
  (0, 251546)	0.06474508206923738
  (0, 183844)	0.11021586662227333
  (0, 262766)	0.09359331900588827
  (0, 196438)	0.1022354174029804
  (0, 2631)	0.08949635371356747
  (0, 164735)	0.12311572162921224
  (0, 97792)	0.04949812442379092
  (0, 51353)	0.08821783878775386
  (0, 36664)	0.0673947503272234
  :	:
  (5170, 201195)	0.071251454910

In [67]:
print(X.shape)

(5171, 263581)


In [68]:
from sklearn.model_selection import train_test_split
X_train , X_test , y_train , y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [69]:
print(X_train.shape)
print(X_test.shape)

(4136, 263581)
(1035, 263581)


In [70]:
#train model
from sklearn.naive_bayes import MultinomialNB
model = MultinomialNB()
model.fit(X_train, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [71]:
#check accuracy
from sklearn.metrics import accuracy_score
y_pred = model.predict(X_test)
acc = accuracy_score(y_test,y_pred)
print("Accuracy:", acc)

Accuracy: 0.8772946859903382


In [72]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.85      1.00      0.92       742
           1       1.00      0.57      0.72       293

    accuracy                           0.88      1035
   macro avg       0.93      0.78      0.82      1035
weighted avg       0.90      0.88      0.87      1035



In [73]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print(cm)

[[742   0]
 [127 166]]


In [77]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

In [78]:
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99       742
           1       0.98      0.98      0.98       293

    accuracy                           0.99      1035
   macro avg       0.99      0.98      0.98      1035
weighted avg       0.99      0.99      0.99      1035

[[736   6]
 [  7 286]]


In [85]:
new_email = [
    "Winner! You have won a luxury vacation package. Click here to confirm your booking."
]

new_email = vectorizer.transform(new_email)

print(model.predict(new_email))

[1]
